In [ ]:
spark

In [ ]:
display(dbutils.fs.ls("abfss://olistdata@storageaccolistdata.dfs.core.windows.net/bronze/"))

%md
## **Reading the Data from filess.io SQL Database**

In [ ]:
base_path = "abfss://olistdata@storageaccolistdata.dfs.core.windows.net/bronze/"

files = dbutils.fs.ls(base_path)

dfs = {}

for file in files:
    clean_name = file.name.replace(".csv", "").replace("_dataset", "")
    var_name = f"{clean_name}_df"

    dfs[var_name] = spark.read.csv(file.path, header=True, inferSchema=True)

for i in dfs:
    print(i)

In [ ]:
dfs['olist_customers_df'].show(5)

## **Reading the Data from Pymongo (filess.io MongoDB Database)**

In [ ]:
# importing module
from pymongo import MongoClient

hostname = "6tyq6c.h.filess.io"
database = "olist_project_nosql_necessary"
port = "27018"
username = "olist_project_nosql_necessary"
password = "870c68751df34a4fa42dc79a36a9cdf1dc7213ac"

uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + port + "/" + database

# Connect with the portnumber and host
client = MongoClient(uri)

# Access database
mydatabase = client[database]

# Store the product_categories data in MongoDB in a variable
collection = mydatabase["product_categories"]

In [ ]:
import pandas as pd

mongo_data = pd.DataFrame(list(collection.find()))
mongo_data.head(10)

## **Cleaning the Data**

In [ ]:
from pyspark.sql.functions import col, to_date, datediff, current_date, to_timestamp, when

# removing duplicates
def clean_dataframe(df, name):
    print("Cleaning" + name)
    return df.dropDuplicates().na.drop('all')

customers_df = clean_dataframe(dfs['olist_customers_df'], "Customers")
customers_df.show(10)

orders_df = clean_dataframe(dfs['olist_orders_df'], "Orders")
orders_df.show(10)

In [ ]:
# convert data columns 

orders_df = orders_df.withColumn("order_purchase_date", to_date(col("order_purchase_timestamp")))\
    .withColumn("order_approved_date", to_date(col("order_approved_at")))\
    .withColumn("order_delivered_carrier_dt", to_date(col("order_delivered_carrier_date")))\
    .withColumn("order_delivered_customer_dt", to_date(col("order_delivered_customer_date")))\
    .withColumn("order_estimated_delivery_dt", to_date(col("order_estimated_delivery_date")))

orders_df.show(10)

In [ ]:
# calculate delivery and time delays 

orders_df = orders_df.withColumn("actual_delivery_time", datediff("order_delivered_customer_date", "order_purchase_timestamp"))\
            .withColumn("estimated_delivery_time", datediff("order_estimated_delivery_date", "order_purchase_timestamp"))\
            .withColumn("delay time", when(col("actual_delivery_time") > col("estimated_delivery_time"), "Yes").otherwise("No"))
display(orders_df)

## **Joining**

In [ ]:
mongo_data.drop("_id", axis=1, inplace=True)

In [ ]:
orders_customers_df = orders_df.join(customers_df, orders_df.customer_id == customers_df.customer_id, "left")

orders_payments_df = orders_customers_df.join(dfs["olist_order_payments_df"], dfs["olist_order_payments_df"].order_id == orders_customers_df.order_id, 'left')

orders_items_df = orders_payments_df.join(dfs["olist_order_items_df"],"order_id", "left")

orders_items_products_df = orders_items_df.join(dfs["olist_products_df"], dfs["olist_products_df"].product_id == orders_items_df.product_id)

final_df_1 = orders_items_products_df.join(dfs["olist_sellers_df"], dfs["olist_sellers_df"].seller_id == orders_items_products_df.seller_id, "left")

mongo_spark_df = spark.createDataFrame(mongo_data)

final_df_2 = final_df_1.join(mongo_spark_df, "product_category_name", "left")


display(final_df_2)

## **Exporting Data to Silver Layer on Azure**

In [ ]:
def remove_duplicate_columns(df):
    columns = df.columns

    seen_columns = set()
    columns_to_drop = []

    for column in columns:
        if column in seen_columns:
            columns_to_drop.append(column)
        else:
            seen_columns.add(column)
    
    df_cleaned = df.drop(*columns_to_drop)
    return df_cleaned

final_df = remove_duplicate_columns(final_df_2)

display(final_df)

In [ ]:
final_df.write.mode("overwrite").parquet("abfss://olistdata@storageaccolistdata.dfs.core.windows.net/silver/")  